# ¿Cuántos trazos bastan? · Demo para la exposición

Carga los modelos ya entrenados por el notebook principal (no entrena nada) y abre la demo en Gradio. Ver `docs/08_demo.md`.

Los modelos entrenados ya vienen en el repositorio (`trabajo/modelos`). La primera vez en cada sesión se crea la caché de prefijos (10 a 20 min en Colab); con `COLAB_ORIGEN = "drive"` queda guardada para la próxima. Con GPU es más fluido, pero en CPU también funciona.

In [ ]:
import os, sys
from pathlib import Path

MODO_RAPIDO = False          # el mismo valor que usaron en el notebook principal
if os.environ.get("QD_MODO_RAPIDO") == "1":
    MODO_RAPIDO = True

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    # Dónde vive el proyecto en Colab (ver docs/00_como_ejecutar.md, sección 1):
    #   "clonar": se clona el repositorio en /content. Se borra al cerrar la
    #             sesión, pero alcanza para estudiar, evaluar y los experimentos
    #             que no reentrenan. No necesita Drive.
    #   "drive" : el repositorio vive en Drive y lo que se guarde sobrevive a los
    #             cortes de Colab. Úsenlo si van a reentrenar algo.
    COLAB_ORIGEN = "clonar"
    REPO = "https://github.com/Faalvial/Quick-Draw.git"
    if COLAB_ORIGEN == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        RAIZ = Path("/content/drive/MyDrive/quickdraw-early")
    else:
        RAIZ = Path("/content/Quick-Draw")
    if not (RAIZ / "quickdraw").exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO, str(RAIZ)], check=True)
    else:
        print("El repositorio ya estaba en", RAIZ, "- para traer cambios: !git -C", RAIZ, "pull")
    !pip install -q gradio
else:
    RAIZ = Path(os.environ.get("QD_RAIZ", Path.cwd().parent))
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)

In [ ]:
from quickdraw.config import Config, config_rapida
from quickdraw import experimentos as E

cfg = Config(ruta_muestra=Path(os.environ.get("QD_MUESTRA", RAIZ / "datos" / "muestra_30")),
             ruta_trabajo=RAIZ / ("trabajo_rapido" if MODO_RAPIDO else "trabajo"))
if MODO_RAPIDO:
    cfg = config_rapida(cfg)

if not (cfg.ruta_modelos / "M2-P_s0" / "modelo.pt").exists():
    raise RuntimeError("No hay modelos entrenados en " + str(cfg.ruta_modelos)
                       + ". Ejecuten primero el notebook principal.")
D = E.preparar_datos(cfg, procesos=os.cpu_count() or 2)   # solo lee las cachés
hp, _ = E.buscar_hiperparametros(cfg, D)                  # solo lee la búsqueda guardada
modelos = E.entrenar_modelos(cfg, D, hp)                  # solo carga los pesos

## Demo

- **Dibujo en vivo:** dibujen en el lienzo; cada trazo actualiza las tres clases más probables de M1, M2 y M2-P.
- **Repetición:** elijan un dibujo real de validación y muevan el control de trazos.

`share=True` crea un enlace público temporal, útil para proyectar desde otra computadora.

In [ ]:
from demo.demo import lanzar
app = lanzar(cfg, D, modelos, semilla=0, share=True,
             abrir=os.environ.get("QD_SIN_ABRIR") != "1")